# Chapter 2 — CUDA C++ vector addition

This notebook builds and runs the vector-addition program for 100003 elements. Run the cells in order on a GPU runtime with `nvcc` available. The delivered notebook contains no GPU execution output.


In [1]:
!nvidia-smi
!nvcc --version
!mkdir -p code

Sat Nov  7 07:44:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile code/chapter02_vector_add_cuda.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t error = (call);                                    \
    if (error != cudaSuccess) {                                    \
        fprintf(stderr, "%s:%d: %s failed: %s\n",                \
                __FILE__, __LINE__, #call,                         \
                cudaGetErrorString(error));                        \
        exit(EXIT_FAILURE);                                        \
    }                                                               \
} while (0)

__global__ void vector_add(const float *a, const float *b,
                           float *c, int n)
{
    const int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] + b[i];
    }
}

int main(int argc, char **argv)
{
    const int n = 100003;
    const size_t bytes = (size_t)n * sizeof(float);
    float *a = (float *)malloc(bytes);
    float *b = (float *)malloc(bytes);
    float *c = (float *)malloc(bytes);
    if (a == NULL || b == NULL || c == NULL) {
        fprintf(stderr, "Could not allocate host arrays.\n");
        free(a); free(b); free(c);
        return EXIT_FAILURE;
    }

    for (int i = 0; i < n; ++i) {
        a[i] = 0.25f * (i % 101);
        b[i] = 0.50f * (i % 37);
        c[i] = -1.0f;  // Detect an output element the kernel did not write.
    }

    float *d_a = NULL;
    float *d_b = NULL;
    float *d_c = NULL;
    CUDA_CHECK(cudaMalloc((void **)&d_a, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_b, bytes));
    CUDA_CHECK(cudaMalloc((void **)&d_c, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    const int threads = 256;
    const int blocks = (n + threads - 1) / threads;
    vector_add<<<blocks, threads>>>(d_a, d_b, d_c, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    int failures = 0;
    for (int i = 0; i < n; ++i) {
        if (c[i] != a[i] + b[i]) {
            ++failures;
        }
    }
    printf("N = %d, threads per block = %d, blocks = %d\n",
           n, threads, blocks);
    printf("Threads without a valid element: %d\n", blocks * threads - n);
    printf("Validation: %s (%d failures among %d elements)\n",
           failures == 0 ? "PASS" : "FAIL", failures, n);

    if (failures == 0 && argc > 1) {
        const int shown = (n < 12) ? n : 12;
        printf("\nFirst %d of %d vector elements:\n", shown, n);
        printf("     i          a[i]          b[i]       GPU c[i]\n");
        for (int i = 0; i < shown; ++i) {
            printf("%6d  %12.6f  %12.6f  %15.6f\n",
                   i, a[i], b[i], c[i]);
        }
        printf("All %d elements were validated.\n", n);
    }

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    free(a); free(b); free(c);
    return failures == 0 ? EXIT_SUCCESS : EXIT_FAILURE;
}


Writing code/chapter02_vector_add_cuda.cu


## Compile

This is the command used in the chapter. `-O2` enables optimization, `-std=c++14` selects C++14, and `-arch=native` targets the GPU assigned to the runtime.


In [3]:
!nvcc -O2 -std=c++14 -arch=native code/chapter02_vector_add_cuda.cu -o vector_add_cuda

## Run and validate

The program checks all 100003 GPU results against the CPU sums. With 256 threads per block, it launches 391 blocks; 93 threads in the final block have no corresponding element.


In [4]:
!./vector_add_cuda

N = 100003, threads per block = 256, blocks = 391
Threads without a valid element: 93
Validation: PASS (0 failures among 100003 elements)


## Display sample values

This reruns the program and prints the first 12 input and output values. All 100003 elements are validated on both runs.


In [5]:
!./vector_add_cuda --show

N = 100003, threads per block = 256, blocks = 391
Threads without a valid element: 93
Validation: PASS (0 failures among 100003 elements)

First 12 of 100003 vector elements:
     i          a[i]          b[i]       GPU c[i]
     0      0.000000      0.000000         0.000000
     1      0.250000      0.500000         0.750000
     2      0.500000      1.000000         1.500000
     3      0.750000      1.500000         2.250000
     4      1.000000      2.000000         3.000000
     5      1.250000      2.500000         3.750000
     6      1.500000      3.000000         4.500000
     7      1.750000      3.500000         5.250000
     8      2.000000      4.000000         6.000000
     9      2.250000      4.500000         6.750000
    10      2.500000      5.000000         7.500000
    11      2.750000      5.500000         8.250000
All 100003 elements were validated.


Save the notebook to retain its cells. The `.cu` file and executable are created in the temporary runtime and can be recreated by rerunning the writing and compilation cells.
